## Imports et configuration WLDA

### Installation des d?pendances (Kaggle-safe)


In [ ]:
import sys
import subprocess
import importlib.util


def ensure_package(import_name: str, pip_spec: str):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_spec}")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            pip_spec,
        ])


# Kaggle-safe: do not force-upgrade torch/cuda.
ensure_package("numpy", "numpy")
ensure_package("scipy", "scipy")
ensure_package("pandas", "pandas")
ensure_package("sklearn", "scikit-learn")
ensure_package("yaml", "pyyaml")
ensure_package("gensim", "gensim")
ensure_package("tqdm", "tqdm")
ensure_package("torch", "torch")

print("Dependencies check done")


In [ ]:
import numpy as np
import pandas as pd
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import scipy.io
import os
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from torch.distributions import Dirichlet


def detect_ppd_root():
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path.cwd().resolve() / "Projet_PPD" / "Projet_PPD",
        Path.cwd().resolve() / "Projet_PPD",
    ]
    for candidate in candidates:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Impossible de localiser un dossier avec data/ et output/.")


PPD_ROOT = detect_ppd_root()
WLDA_ROOT = str(PPD_ROOT / "output" / "WLDA")
DATA_ROOT = str(PPD_ROOT / "data")

SEED = 42
TOPN = 15
DATASETS = ["20NG", "AGNews", "IMDB", "YahooAnswer"]
K_VALUES = [50, 100]

lr = 2e-4
epochs = 200
batch_size = 100
hidden_size = 1024

os.makedirs(WLDA_ROOT, exist_ok=True)
print(f"Dossier WLDA d?di?: {WLDA_ROOT}")


## Fonctions utilitaires

In [8]:
import os
import random
import numpy as np
import torch
import scipy.sparse as sp

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # comportement déterministe
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_dataset(name: str):
    """
    Charge un dataset au format BoW.

    Returns:
        train_bow (np.ndarray)
        test_bow  (np.ndarray)
        vocab     (list[str])
        labels    (np.ndarray)
    """
    path = os.path.join(DATA_ROOT, name)

    train_bow = sp.load_npz(os.path.join(path, "train_bow.npz")).toarray()
    test_bow  = sp.load_npz(os.path.join(path, "test_bow.npz")).toarray()

    with open(os.path.join(path, "vocab.txt"), encoding="utf-8") as f:
        vocab = [w.strip() for w in f]

    labels = np.loadtxt(os.path.join(path, "train_labels.txt"), dtype=int)

    return train_bow, test_bow, vocab, labels

SEED = 42
set_seed(SEED)

print("Fonctions prêtes ")


Fonctions prêtes 


## Modèle WLDA

In [9]:
class WLDA(nn.Module):
    def __init__(self, vocab_size, num_topics, hidden_size=512):  # ↑512
        super().__init__()
        self.num_topics = num_topics
        self.vocab_size = vocab_size
        
        # Encoder: BOW -> alpha
        self.encoder = nn.Sequential(
            nn.Linear(vocab_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_topics),
            nn.Softplus() # alpha > 0
        )
        
        # Beta init plus petite
        self.beta = nn.Parameter(torch.randn(num_topics, vocab_size) * 0.001)  # ↓0.01→0.001
        
    def encode(self, x):
        alpha = self.encoder(x) + 1e-6
        return alpha
    
    def reparameterize(self, alpha):
        dist = Dirichlet(alpha)
        theta = dist.rsample()
        return theta
    
    def decode(self, theta):
        logits = torch.matmul(theta, self.beta)
        return F.log_softmax(logits, dim=-1)
    
    def forward(self, x):
        alpha = self.encode(x)
        theta = self.reparameterize(alpha)
        recon_loss = -(x * self.decode(theta)).sum(dim=-1).mean()
        
        # KL plus faible
        kl_loss = torch.distributions.kl_divergence(
            Dirichlet(alpha), 
            Dirichlet(torch.ones_like(alpha))
        ).mean()
        
        return recon_loss + 0.05 * kl_loss  # ↓0.1→0.05

    def get_beta(self):
        return F.softmax(self.beta, dim=-1)
    
    def get_theta(self, bow):
        alpha = self.encode(bow)
        dist = Dirichlet(alpha)
        return dist.rsample()

print("Modèle WLDA defini")


Modele WLDA defini


## Fonctions métriques

In [10]:
import collections
import sklearn.metrics

def topic_diversity(topics):
    """TD = proportion mots exclusifs"""
    word_freq = collections.Counter()
    for topic in topics:
        word_freq.update(topic)
    exclusifs = sum(1 for count in word_freq.values() if count == 1)
    return exclusifs / len(word_freq) if word_freq else 0.0

def purity_score(y_true, y_pred):
    """Purity standard"""
    total = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        labels, counts = np.unique(y_true[idx], return_counts=True)
        total += counts.max()
    return total / len(y_true)

def compute_nmi(theta, labels):
    """NMI standard"""
    pred = theta.argmax(axis=1)
    return sklearn.metrics.normalized_mutual_info_score(labels, pred)


## Entraînement WLDA 

In [11]:
def train_one_wlda(dataset, K, seed=42):
    set_seed(seed)
    total_start = time.perf_counter()

    dataset_dir = f"{WLDA_ROOT}/{dataset}"
    os.makedirs(dataset_dir, exist_ok=True)

    model_path = f"{dataset_dir}/WLDA_{dataset}_K{K}_seed{seed}.pt"
    mat_path = f"{dataset_dir}/{dataset}_WLDA_K{K}_seed{seed}.mat"

    if os.path.exists(mat_path):
        print(f"Deja existant: {mat_path}")
        return {'mat_path': mat_path, 'timing': None}

    print(f"WLDA | {dataset} | K={K}")

    # Données
    train_bow, test_bow, vocab, _ = load_dataset(dataset)
    V = train_bow.shape[1]

    # Modèle (CPU)
    model = WLDA(V, K, hidden_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Dataset (CPU)
    X_train_t = torch.tensor(train_bow, dtype=torch.float32)
    loader = DataLoader(
        TensorDataset(X_train_t),
        batch_size=batch_size,
        shuffle=True
    )

    # Entraînement
    train_start = time.perf_counter()
    epoch_bar = tqdm(range(epochs), desc=f"WLDA_{dataset}_K{K}")
    for epoch in epoch_bar:
        model.train()
        total_loss = 0.0

        for (batch,) in loader:
            optimizer.zero_grad()
            loss = model(batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        epoch_bar.set_postfix(loss=f"{total_loss / len(loader):.2f}")

    train_seconds = time.perf_counter() - train_start

    # Extraire beta / theta (CPU)
    model.eval()
    with torch.no_grad():
        beta_np = model.get_beta().numpy()

        X_train = torch.tensor(train_bow, dtype=torch.float32)
        train_theta = model.get_theta(X_train).numpy()

        X_test = torch.tensor(test_bow, dtype=torch.float32)
        test_theta = model.get_theta(X_test).numpy()

    scipy.io.savemat(mat_path, {
        "beta": beta_np.astype(np.float32),
        "train_theta": train_theta.astype(np.float32),
        "test_theta": test_theta.astype(np.float32)
    })

    torch.save(model.state_dict(), model_path)

    total_seconds = time.perf_counter() - total_start
    timing_row = {
        'model': 'WLDA', 'dataset': dataset, 'K': int(K), 'seed': int(seed), 'device': 'cpu',
        'phase': 'train', 'train_seconds': float(train_seconds), 'total_seconds': float(total_seconds),
        'train_minutes': float(train_seconds / 60.0), 'total_minutes': float(total_seconds / 60.0),
    }

    timing_path = f"{dataset_dir}/{dataset}_WLDA_K{K}_seed{seed}_timing.csv"
    pd.DataFrame([timing_row]).to_csv(timing_path, index=False)

    print(f"Sauvegarde: {mat_path}")
    print(f"Timing: {timing_path} | train={train_seconds:.2f}s total={total_seconds:.2f}s")

    return {'mat_path': mat_path, 'timing': timing_row, 'timing_path': timing_path}


## Coh?rence C_V avec Gensim


## Metrics TD/Purity/NMI

In [ ]:
def compute_metrics_wlda(dataset, K, seed=42, topn=15):
    dataset_dir = f"{WLDA_ROOT}/{dataset}"
    mat_path = f"{dataset_dir}/{dataset}_WLDA_K{K}_seed{seed}.mat"

    if not os.path.isfile(mat_path):
        print('.mat manquant')
        return None

    loaded = scipy.io.loadmat(mat_path)
    beta = loaded['beta']
    train_theta = loaded['train_theta']

    _, _, vocab, labels = load_dataset(dataset)

    topics = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        topics.append([vocab[i] for i in top_idx])
    td = topic_diversity(topics)

    y_pred = train_theta.argmax(axis=1)
    pur = purity_score(labels, y_pred)
    nmi = compute_nmi(train_theta, labels)

    print(f"WLDA {dataset} K={K}: TD={td:.6f}, Purity={pur:.6f}, NMI={nmi:.6f}")

    out_path = f"{dataset_dir}/metrics_{dataset}_WLDA_K{K}_seed{seed}.txt"
    with open(out_path, 'w') as f:
        f.write(f"dataset={dataset}\nmodel=WLDA\nK={K}\nseed={seed}\n")
        f.write(f"TD={td:.6f}\nPurity={pur:.6f}\nNMI={nmi:.6f}\n")

    print(f"Metrics: {out_path}")
    return out_path


##  Entraînement tous modèles

In [13]:
print("=" * 60)
print("ENTRAINEMENT WLDA - 4 BASES")
print("=" * 60)

total_processed = 0
total_skipped = 0
total_models = len(DATASETS) * len(K_VALUES)
training_time_rows = []

with tqdm(total=total_models, desc="WLDA Global") as pbar:
    for dataset in DATASETS:
        print(f"\n{'='*50}")
        print(f"BASE: {dataset}")

        with tqdm(total=len(K_VALUES), desc=f"{dataset}", leave=False) as pbar_k:
            for K in K_VALUES:
                dataset_dir = f"{WLDA_ROOT}/{dataset}"
                mat_path = f"{dataset_dir}/{dataset}_WLDA_K{K}_seed{SEED}.mat"

                print(f"\n--- {dataset} K={K} ---")

                if os.path.exists(mat_path):
                    print(f"SKIP (deja calcule)")
                    total_skipped += 1
                else:
                    out = train_one_wlda(dataset, K, seed=SEED)
                    if isinstance(out, dict) and out.get('timing') is not None:
                        training_time_rows.append(out['timing'])
                    total_processed += 1

                pbar_k.update(1)
                pbar.update(1)
                pbar.set_postfix({'processed': total_processed, 'skipped': total_skipped})

print(f"\nTraitees: {total_processed}, Sautees: {total_skipped}")

if training_time_rows:
    df_train_times = pd.DataFrame(training_time_rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df_train_times[['dataset', 'K', 'device', 'train_seconds', 'total_seconds']])

    time_csv = f"{WLDA_ROOT}/WLDA_training_times.csv"
    df_train_times.to_csv(time_csv, index=False)
    print(f"Saved training times: {time_csv}")


ENTRAINEMENT WLDA - 4 BASES


WLDA Global:   0%|          | 0/8 [00:00<?, ?it/s]


BASE: 20NG


20NG:   0%|          | 0/2 [00:00<?, ?it/s]


--- 20NG K=50 ---
SKIP (deja calcule)

--- 20NG K=100 ---
SKIP (deja calcule)

BASE: AGNews


...


--- AGNews K=50 ---
SKIP (deja calcule)

--- AGNews K=100 ---
SKIP (deja calcule)

BASE: IMDB


IMDB:   0%|          | 0/2 [00:00<?, ?it/s]


--- IMDB K=50 ---
SKIP (deja calcule)

--- IMDB K=100 ---
SKIP (deja calcule)

BASE: YahooAnswer


YahooAnswer:   0%|          | 0/2 [00:00<?, ?it/s]


--- YahooAnswer K=50 ---
SKIP (deja calcule)

--- YahooAnswer K=100 ---
SKIP (deja calcule)

Traitees: 0, Sautees: 8


## Toutes métriques

In [8]:
for dataset in DATASETS:
    print(f"\n{'='*50}")
    print(f"BASE: {dataset}")
    
    for K in K_VALUES:
        print(f"\n--- {dataset} K={K} ---")
        compute_metrics_wlda(dataset, K)   # affiche TD/Purity/NMI



BASE: 20NG

--- 20NG K=50 ---
WLDA 20NG K=50: TD=0.486486, Purity=0.168022, NMI=0.258929
Metrics: /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/WLDA/20NG/metrics_20NG_WLDA_K50_seed42.txt

--- 20NG K=100 ---
WLDA 20NG K=100: TD=0.410256, Purity=0.188262, NMI=0.254451
Metrics: /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/WLDA/20NG/metrics_20NG_WLDA_K100_seed42.txt

BASE: AGNews

--- AGNews K=50 ---
WLDA AGNews K=50: TD=0.459459, Purity=0.657900, NMI=0.288064
Metrics: /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/WLDA/AGNews/metrics_AGNews_WLDA_K50_seed42.txt

--- AGNews K=100 ---
WLDA AGNews K=100: TD=0.050000, Purity=0.316500, NMI=0.009220
Metrics: /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/WLDA/AGNews/metrics_AGNews_WLDA_K100_seed42.txt

BASE: IMDB

--- IMDB K=50 ---
WLDA IMDB K=50: TD=0.306452, Purity=0.683880, NMI=0.075813
Metrics: /Users/hydersidra/Documents/Master/PPD/ECRTM_runs/WLDA/IMDB/metrics_IMDB_WLDA_K50_seed42.txt

--- IMDB K=100 ---
WLDA IMDB K=100: TD=

In [17]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
import pandas as pd
# ⚠️ suppose que tu as déjà:
# - la fonction load_dataset(name) qui renvoie (trainbow, testbow, vocab, labels) [file:391]
# - les matrices beta déjà sauvées dans les .mat par train_one_wlda_dataset [file:391]

def compute_gensim_cv_wlda_dataset(dataset, K, topn=15):
    """
    Calcule C_V (Gensim-like) pour WLDA avec Gensim, en utilisant les vrais textes du corpus.
    """
    print(f"\n=== WLDA {dataset} K={K} — C_V (gensim) ===")

    # 1) Charger BoW + vocab
    train_bow, test_bow, vocab, labels = load_dataset(dataset)  

    # 2) Construire textes tokenisés à partir de train_bow
    tokenized_texts = []
    for doc in train_bow:
        words = [vocab[i] for i, freq in enumerate(doc) if freq > 0]
        if len(words) >= 2:
            tokenized_texts.append(words)

    print(f"Corpus pour Gensim: {len(tokenized_texts)} docs, {len(vocab)} mots")

    # 3) Charger beta depuis le .mat WLDA
    dataset_dir = f"{WLDA_ROOT}/{dataset}"
    mat_path = f"{dataset_dir}/{dataset}_WLDA_K{K}_seed{SEED}.mat"
    if not os.path.isfile(mat_path):
        print(f"❌ Pas de .mat pour {dataset} K={K}: {mat_path}")
        return None

    loaded = scipy.io.loadmat(mat_path)
    beta = loaded["beta"]  # shape (K, V)
    print(f"beta: {beta.shape}")

    # 4) Construire les topics (listes de mots) pour Gensim
    wlda_topics = []
    for k in range(beta.shape[0]):
        top_idx = beta[k].argsort()[-topn:][::-1]  # top-n indices
        topic_words = [vocab[i] for i in top_idx if i < len(vocab)]
        if len(topic_words) >= 2:
            wlda_topics.append(topic_words)

    print(f"{len(wlda_topics)} topics construits (top{topn})")

    # 5) Dictionnaire + modèle de cohérence C_V
    dictionary = Dictionary(tokenized_texts)
    cm = CoherenceModel(
        topics=wlda_topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence="c_v",   # C_V Gensim-like
    )

    cv_score = cm.get_coherence()
    print(f"✅ C_V (Gensim) WLDA {dataset} K={K} = {cv_score:.4f}")
    return cv_score


# Exemple : calculer C_V pour tous datasets / K
wlda_cv_results = []
for dataset in DATASETS:      # ["20NG", "AGNews", "IMDB", "YahooAnswer"] [file:391]
    for K in K_VALUES:         # [50, 100] [file:391]
        cv = compute_gensim_cv_wlda_dataset(dataset, K, topn=TOPN)  # TOPN=15 [file:391]
        if cv is not None:
            wlda_cv_results.append({"Dataset": dataset, "K": K, "C_V": cv})

wlda_cv_df = pd.DataFrame(wlda_cv_results)
display(wlda_cv_df.pivot(index="Dataset", columns="K", values="C_V").round(4))



=== WLDA 20NG K=50 — C_V (gensim) ===
Corpus pour Gensim: 11314 docs, 5000 mots
beta: (50, 5000)
50 topics construits (top15)
✅ C_V (Gensim) WLDA 20NG K=50 = 0.5225

=== WLDA 20NG K=100 — C_V (gensim) ===
Corpus pour Gensim: 11314 docs, 5000 mots
beta: (100, 5000)
100 topics construits (top15)
✅ C_V (Gensim) WLDA 20NG K=100 = 0.5298

=== WLDA AGNews K=50 — C_V (gensim) ===
Corpus pour Gensim: 9999 docs, 5000 mots
beta: (50, 5000)
50 topics construits (top15)
✅ C_V (Gensim) WLDA AGNews K=50 = 0.2606

=== WLDA AGNews K=100 — C_V (gensim) ===
Corpus pour Gensim: 9999 docs, 5000 mots
beta: (100, 5000)
100 topics construits (top15)
✅ C_V (Gensim) WLDA AGNews K=100 = 0.2611

=== WLDA IMDB K=50 — C_V (gensim) ===
Corpus pour Gensim: 25000 docs, 5000 mots
beta: (50, 5000)
50 topics construits (top15)
✅ C_V (Gensim) WLDA IMDB K=50 = 0.2635

=== WLDA IMDB K=100 — C_V (gensim) ===
Corpus pour Gensim: 25000 docs, 5000 mots
beta: (100, 5000)
100 topics construits (top15)
✅ C_V (Gensim) WLDA IMDB K

K,50,100
Dataset,,
20NG,0.5225,0.5298
AGNews,0.2606,0.2611
IMDB,0.2635,0.2830
YahooAnswer,0.5856,0.6469


In [19]:
import pandas as pd

# Données du Papier Original (Source: image de gauche)
paper_wlda = {
    "20NG": {
        "K50":  {"C_V": 0.523, "TD": 0.486, "Purity": 0.168, "NMI": 0.259},
        "K100": {"C_V": 0.530, "TD": 0.410, "Purity": 0.188, "NMI": 0.254}
    },
    "IMDB": {
        "K50":  {"C_V": 0.264, "TD": 0.306, "Purity": 0.684, "NMI": 0.076},
        "K100": {"C_V": 0.283, "TD": 0.386, "Purity": 0.683, "NMI": 0.065}
    },
    "YahooAnswer": {
        "K50":  {"C_V": 0.586, "TD": 0.394, "Purity": 0.286, "NMI": 0.138},
        "K100": {"C_V": 0.647, "TD": 0.333, "Purity": 0.348, "NMI": 0.161}
    },
    "AGNews": {
        "K50":  {"C_V": 0.261, "TD": 0.459, "Purity": 0.658, "NMI": 0.288},
        "K100": {"C_V": 0.261, "TD": 0.050, "Purity": 0.317, "NMI": 0.009}
    }
}

# Vos résultats de Reproduction (Source: image de droite)
reproduction_wlda = {
    "20NG": {
        "K50":  {"C_V": 0.378, "TD": 0.375, "Purity": 0.233, "NMI": 0.157},
        "K100": {"C_V": 0.369, "TD": 0.273, "Purity": 0.292, "NMI": 0.207}
    },
    "IMDB": {
        "K50":  {"C_V": 0.311, "TD": 0.053, "Purity": 0.589, "NMI": 0.011},
        "K100": {"C_V": 0.320, "TD": 0.069, "Purity": 0.602, "NMI": 0.013}
    },
    "YahooAnswer": {
        "K50":  {"C_V": 0.333, "TD": 0.322, "Purity": 0.255, "NMI": 0.084},
        "K100": {"C_V": 0.338, "TD": 0.292, "Purity": 0.303, "NMI": 0.127}
    },
    "AGNews": {
        "K50":  {"C_V": 0.384, "TD": 0.410, "Purity": 0.580, "NMI": 0.151},
        "K100": {"C_V": 0.378, "TD": 0.323, "Purity": 0.653, "NMI": 0.188}
    }
}

data = []
for dataset in ["20NG", "IMDB", "YahooAnswer", "AGNews"]:
    for k_str in ["K50", "K100"]:
        p, r = paper_wlda[dataset][k_str], reproduction_wlda[dataset][k_str]
        row = {"Dataset": dataset, "K": int(k_str[1:])}
        for m in ["C_V", "TD", "Purity", "NMI"]:
            row[f"{m}_Repro"]  = r[m]  # Reproduction à gauche
            row[f"{m}_Papier"] = p[m]  # Papier à droite
            row[f"{m}_Ecart"]  = round(r[m] - p[m], 3)
        data.append(row)

df = pd.DataFrame(data).set_index(["Dataset", "K"])

def color_ecart(val):
    if not isinstance(val, (int, float)): return ""
    if abs(val) < 0.02: return "background-color: #c6efce; color: #276221" # Vert (proche)
    if abs(val) < 0.05: return "background-color: #ffeb9c; color: #9c6500" # Jaune (moyen)
    return "background-color: #ffc7ce; color: #9c0006" # Rouge (éloigné)

ecart_cols = [c for c in df.columns if "Ecart" in c]

# Affichage du tableau stylisé
df.style\
    .format("{:.3f}")\
    .map(color_ecart, subset=ecart_cols)\
    .set_caption("WLDA — Reproduction vs Papier Original (Écarts)")